In [ ]:
# set up indicator ID and get scenarios from database
#IndID= "310" #Indicator ID (Exposure = 2 + 0X where X is the Exposure Indicator number, Peligro= 1 +0x, VSB= 3 +0x, VSS= 4 +0x, VCA= 5 +0x)
conn = sqlite3.connect(db_path)

scenarios_df = pd.read_sql_query(
    """
    SELECT ScnID, ScnName
    FROM ScnMod
    """,
    conn
)

indicators_df = pd.read_sql_query(
    """
    SELECT IndID, TextID
    FROM Indicators
    WHERE SIG_Type = 'WaterALLOC DB'
    """,
    conn
)

conn.close()

# For now: only baseline and first future
#scenario_ids = scenarios_df.loc[
#    scenarios_df['ScnID'].isin([1, 2]), 'ScnID'
#].tolist()

# for all scenarios:
print("Available scenarios in the indicators DB:")
scenario_ids = scenarios_df['ScnID'].tolist()

scenarios_df

Check what scenarios are available in the WaterALLOC database and the corresponding indicator scenario.

In [ ]:
# Connect to WaterALLOC database
conn_wa = sqlite3.connect(wateralloc_db)

# Query available scenarios
scenarios_query = """
SELECT WaScnID, Scenario, IndScnName
FROM Scenarios
ORDER BY Scenario
"""

wa_scenarios_df = pd.read_sql_query(scenarios_query, conn_wa)

print("Available scenarios in WaterALLOC DB:")
for s in wa_scenarios_df.iterrows():
    print(f" - {s[1].Scenario} -> {s[1].IndScnName}")
           
conn_wa.close()


#### Find errors matching WaterALLOC scenarios to the indicators

In [ ]:
# Find issue with not matching scenario names with the indicator database
# if IndScnName is null, print an error message
mismatched_scenarios = wa_scenarios_df[wa_scenarios_df["IndScnName"].isnull()]
if not mismatched_scenarios.empty:
    print("\nError: The following WaterALLOC scenarios do not have matching indicator scenario names:")
    for index, row in mismatched_scenarios.iterrows():
        print(f" - WaScnID: {row['WaScnID']}, Scenario: {row['Scenario']}")
        
# if the names in IndScnName do not match any of the ScnName in scenarios_df, print an error message
for index, row in wa_scenarios_df.iterrows():
    if pd.notnull(row["IndScnName"]):
        if row["IndScnName"] not in scenarios_df["ScnName"].values:
            print(f"\nError: WaterALLOC scenario '{row['IndScnName']}' does not match any ScnName in indicator database.")  
            